# 06 — Active Label Cleaning

## Objective

Rank coin annotations whose predicted class differs from the original label using the Active Label Cleaning score `Phi = CE - H`.

## Motivation

The original method assumes single-label classification, while this dataset contains object-detection annotations. Treating each annotated coin crop as a classification sample makes the method applicable while preserving human review as the only source of label decisions. High-scoring rows are review candidates, not confirmed errors.

## Inputs

- `curation/outputs/02-annotation-audit/annotations_normalized.csv`
- Images from the configured image directory when training, inference, or visualization is required.
- Compatible checkpoint or posterior cache when available.

## Outputs

- `curation/outputs/06-active-label-cleaning/alc_selector_training_history.csv`
- `curation/outputs/06-active-label-cleaning/model/alc_vanilla_resnet50.pt`
- `curation/outputs/06-active-label-cleaning/cache/alc_selector_posteriors.npy`
- `curation/outputs/06-active-label-cleaning/alc_label_candidates.csv`
- `curation/outputs/06-active-label-cleaning/manifest.yaml`

Resume order is candidates, posterior cache, checkpoint, and finally training. Drive images are copied to `/content` only when an image-dependent operation is required.


In [ ]:
# Environment-specific setup
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

ROOT = base_folder.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from curation.common import load_config, prepare_dataset, stage_output_dir

CONFIG = load_config(ROOT)
DATASET = prepare_dataset(ROOT)
IMAGES_DIR = DATASET["images_dir"]
ANNOTATIONS_DIR = DATASET["annotations_dir"]

from curation.common import (
    crop_bbox, load_manifest, ordered_rows_fingerprint, sha256_file,
    stable_key, write_manifest,
)
import json
import math
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

STAGE = "06-active-label-cleaning"
STAGE_DIR = stage_output_dir(STAGE, ROOT)
MODEL_DIR = STAGE_DIR / "model"
CACHE_DIR = STAGE_DIR / "cache"
MODEL_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
ANNOTATIONS_PATH = stage_output_dir("02-annotation-audit", ROOT, create=False) / "annotations_normalized.csv"
CHECKPOINT_PATH = MODEL_DIR / "alc_vanilla_resnet50.pt"
HISTORY_PATH = STAGE_DIR / "alc_selector_training_history.csv"
POSTERIOR_PATH = CACHE_DIR / "alc_selector_posteriors.npy"
CANDIDATES_PATH = STAGE_DIR / "alc_label_candidates.csv"

In [ ]:
alc_config = CONFIG["alc"]
training_config = alc_config["training"]
coin_labels = list(CONFIG["classes"]["coins"])
class_to_index = {label: index for index, label in enumerate(coin_labels)}
index_to_class = dict(enumerate(coin_labels))

seed = int(CONFIG["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

annotations = pd.read_csv(ANNOTATIONS_PATH, dtype={"label": str})
samples = annotations[annotations["label"].isin(coin_labels)].dropna(subset=["bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]).copy().reset_index(drop=True)
samples["sample_id"] = np.arange(len(samples), dtype=int)
samples["original_label"] = samples["label"]
fingerprint_columns = ["annotation_id", "relative_path", "shape_index", "original_label", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
sample_fingerprint = ordered_rows_fingerprint(samples, fingerprint_columns)
config_hash = sha256_file(ROOT / "curation" / "config.yaml")
manifest = load_manifest(STAGE, ROOT)
print("Samples:", len(samples), "| fingerprint:", sample_fingerprint)

In [ ]:
probability_columns = [f"probability_{label}" for label in coin_labels]
candidate_columns = [
    "candidate_key", "priority_rank", "sample_id", "annotation_id", "relative_path", "shape_index",
    "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax", "original_label", "predicted_label",
    *probability_columns, "predicted_confidence", "probability_original_label",
    "noisiness_ce", "ambiguity_entropy", "alc_phi",
]

def manifest_has_artifact(relative_path):
    return relative_path in manifest.get("artifacts", [])

def common_manifest_is_compatible():
    return (
        manifest.get("configuration", {}).get("sha256") == config_hash
        and manifest.get("compatibility", {}).get("samples", {}).get("fingerprint") == sample_fingerprint
        and manifest.get("compatibility", {}).get("samples", {}).get("count") == len(samples)
    )

def candidates_are_valid():
    if not CANDIDATES_PATH.is_file() or not manifest_has_artifact(CANDIDATES_PATH.name) or not common_manifest_is_compatible():
        return False
    candidate_metadata = manifest.get("compatibility", {}).get("candidates", {})
    if candidate_metadata.get("schema") != candidate_columns or not candidate_metadata.get("only_prediction_divergences"):
        return False
    try:
        frame = pd.read_csv(CANDIDATES_PATH, dtype={"original_label": str, "predicted_label": str})
    except (OSError, ValueError, pd.errors.ParserError):
        return False
    if list(frame.columns) != candidate_columns or frame["candidate_key"].duplicated().any():
        return False
    if not (frame["original_label"] != frame["predicted_label"]).all():
        return False
    sample_ids = frame["sample_id"].to_numpy(int)
    if np.any((sample_ids < 0) | (sample_ids >= len(samples))):
        return False
    expected_rows = samples.iloc[sample_ids].reset_index(drop=True)
    if not np.array_equal(frame["annotation_id"].astype(str), expected_rows["annotation_id"].astype(str)) or not np.array_equal(frame["relative_path"].astype(str), expected_rows["relative_path"].astype(str)) or not np.array_equal(frame["original_label"].astype(str), expected_rows["original_label"].astype(str)):
        return False
    expected_keys = [stable_key("alc", annotation_id, relative, shape_index) for annotation_id, relative, shape_index in frame[["annotation_id", "relative_path", "shape_index"]].itertuples(index=False, name=None)]
    if not np.array_equal(frame["candidate_key"].astype(str), expected_keys):
        return False
    probabilities = frame[probability_columns].to_numpy(float)
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        return False
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=float(alc_config["posterior_sum_atol"])):
        return False
    expected_prediction = np.asarray(coin_labels)[probabilities.argmax(axis=1)]
    expected_confidence = probabilities.max(axis=1)
    original_index = np.array([class_to_index[label] for label in frame["original_label"]])
    original_probability = probabilities[np.arange(len(frame)), original_index]
    ce = -np.log(np.clip(original_probability, 1e-12, 1.0))
    entropy = -np.sum(np.clip(probabilities, 1e-12, 1.0) * np.log(np.clip(probabilities, 1e-12, 1.0)), axis=1)
    return (
        np.array_equal(frame["priority_rank"].to_numpy(), np.arange(1, len(frame) + 1))
        and frame["alc_phi"].is_monotonic_decreasing
        and frame["sample_id"].tolist() == frame.sort_values(["alc_phi", "sample_id"], ascending=[False, True])["sample_id"].tolist()
        and np.array_equal(frame["predicted_label"].to_numpy(str), expected_prediction)
        and np.allclose(frame["predicted_confidence"], expected_confidence)
        and np.allclose(frame["probability_original_label"], original_probability)
        and np.allclose(frame["noisiness_ce"], ce)
        and np.allclose(frame["ambiguity_entropy"], entropy)
        and np.allclose(frame["alc_phi"], ce - entropy)
    )

def posterior_is_valid(array):
    metadata = manifest.get("compatibility", {}).get("posterior", {})
    return (
        POSTERIOR_PATH.is_file() and manifest_has_artifact("cache/alc_selector_posteriors.npy") and common_manifest_is_compatible()
        and metadata.get("sample_fingerprint") == sample_fingerprint
        and metadata.get("shape") == [len(samples), len(coin_labels)]
        and metadata.get("dtype") == str(array.dtype)
        and float(metadata.get("sum_atol", -1)) == float(alc_config["posterior_sum_atol"])
        and list(array.shape) == [len(samples), len(coin_labels)]
        and np.isfinite(array).all() and np.all((array >= 0) & (array <= 1))
        and np.allclose(array.sum(axis=1), 1.0, atol=float(alc_config["posterior_sum_atol"]))
    )

In [ ]:
image_size = int(alc_config["image_size"])
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)), transforms.RandomHorizontalFlip(0.5),
    transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), shear=5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
inference_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

ACTIVE_IMAGES_DIR = IMAGES_DIR
def prepare_fast_image_access():
    global ACTIVE_IMAGES_DIR
    if str(IMAGES_DIR).startswith("/content/drive/"):
        local = Path("/content") / IMAGES_DIR.name
        if not local.exists():
            shutil.copytree(IMAGES_DIR, local)
        ACTIVE_IMAGES_DIR = local
    return ACTIVE_IMAGES_DIR

class CoinCrops(Dataset):
    def __init__(self, frame, transform):
        self.frame, self.transform = frame.reset_index(drop=True), transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(ACTIVE_IMAGES_DIR / row["relative_path"]).convert("RGB") as image:
            crop = crop_bbox(image, row, float(alc_config["crop_padding_fraction"]))
        if crop is None:
            raise ValueError(f"Empty crop for sample {index}")
        return self.transform(crop), class_to_index[row["original_label"]], int(row["sample_id"])

def build_selector():
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(coin_labels))
    return model.to(device)

def model_attestation_is_current():
    metadata = manifest.get("compatibility", {}).get("model", {})
    return bool(CHECKPOINT_PATH.is_file() and manifest_has_artifact("model/alc_vanilla_resnet50.pt") and common_manifest_is_compatible() and metadata.get("load_validated") and metadata.get("architecture") == alc_config["architecture"] and metadata.get("classes") == coin_labels and metadata.get("image_size") == image_size and metadata.get("training") == training_config)

def checkpoint_is_valid():
    if not model_attestation_is_current():
        return False, None
    try:
        payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
        state = payload["model_state_dict"]
        signature_ok = state["fc.weight"].shape == (len(coin_labels), 2048) and "layer4.2.conv3.weight" in state
        payload_ok = list(payload.get("coin_labels", coin_labels)) == coin_labels and int(payload.get("image_size", image_size)) == image_size
        return bool(signature_ok and payload_ok), payload
    except Exception:
        return False, None

def train_selector():
    prepare_fast_image_access()
    loader = DataLoader(CoinCrops(samples, train_transform), batch_size=int(training_config["batch_size"]), shuffle=True, num_workers=int(training_config["num_workers"]), pin_memory=torch.cuda.is_available())
    model = build_selector()
    optimizer = torch.optim.SGD(model.parameters(), lr=float(training_config["base_lr"]), momentum=float(training_config["momentum"]), nesterov=True, weight_decay=float(training_config["weight_decay"]))
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(training_config["lr_milestones"]), gamma=float(training_config["lr_decay"]))
    criterion, history = nn.CrossEntropyLoss(), []
    for epoch in range(int(training_config["epochs"])):
        model.train(); running_loss = 0.0; seen = 0
        for inputs, labels, _ in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(); loss = criterion(model(inputs), labels); loss.backward(); optimizer.step()
            running_loss += float(loss.item()) * len(inputs); seen += len(inputs)
        scheduler.step()
        history.append({"epoch": epoch + 1, "loss": running_loss / seen, "lr": optimizer.param_groups[0]["lr"]})
    payload = {"model_state_dict": model.state_dict(), "architecture": alc_config["architecture"], "coin_labels": coin_labels, "image_size": image_size, "training_parameters": dict(training_config), "sample_fingerprint": sample_fingerprint, "history": history}
    torch.save(payload, CHECKPOINT_PATH)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    return model, payload

def infer(model):
    prepare_fast_image_access()
    loader = DataLoader(CoinCrops(samples, inference_transform), batch_size=int(training_config["batch_size"]), shuffle=False, num_workers=int(training_config["num_workers"]), pin_memory=torch.cuda.is_available())
    result = np.zeros((len(samples), len(coin_labels)), dtype=np.float64)
    model.eval()
    with torch.no_grad():
        for inputs, _, sample_ids in loader:
            probabilities = torch.softmax(model(inputs.to(device)), dim=1).cpu().numpy()
            result[np.asarray(sample_ids, dtype=int)] = probabilities
    if not np.isfinite(result).all() or not np.allclose(result.sum(axis=1), 1.0, atol=float(alc_config["posterior_sum_atol"])):
        raise RuntimeError("Calculated posterior failed validation.")
    np.save(POSTERIOR_PATH, result)
    return result

In [ ]:
def build_candidates(posterior):
    original_index = np.array([class_to_index[label] for label in samples["original_label"]])
    predicted_index = posterior.argmax(axis=1)
    predicted_label = np.asarray(coin_labels)[predicted_index]
    probability_original = posterior[np.arange(len(samples)), original_index]
    ce = -np.log(np.clip(probability_original, 1e-12, 1.0))
    clipped = np.clip(posterior, 1e-12, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    phi = ce - entropy
    frame = samples[["sample_id", "annotation_id", "relative_path", "shape_index", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax", "original_label"]].copy()
    frame["predicted_label"] = predicted_label
    for index, label in enumerate(coin_labels):
        frame[f"probability_{label}"] = posterior[:, index]
    frame["predicted_confidence"] = posterior.max(axis=1)
    frame["probability_original_label"] = probability_original
    frame["noisiness_ce"] = ce
    frame["ambiguity_entropy"] = entropy
    frame["alc_phi"] = phi
    frame = frame[frame["original_label"] != frame["predicted_label"]].sort_values(["alc_phi", "sample_id"], ascending=[False, True]).reset_index(drop=True)
    frame.insert(0, "priority_rank", np.arange(1, len(frame) + 1))
    frame.insert(0, "candidate_key", [stable_key("alc", annotation_id, relative, shape_index) for annotation_id, relative, shape_index in frame[["annotation_id", "relative_path", "shape_index"]].itertuples(index=False, name=None)])
    return frame[candidate_columns], predicted_label

def save_manifest(posterior, candidates, predicted_label):
    artifacts = [path for path in (HISTORY_PATH, CHECKPOINT_PATH, POSTERIOR_PATH, CANDIDATES_PATH) if path.is_file()]
    confusion = pd.crosstab(pd.Series(samples["original_label"], name="original"), pd.Series(predicted_label, name="predicted")).reindex(index=coin_labels, columns=coin_labels, fill_value=0)
    distribution = candidates["original_label"].value_counts().reindex(coin_labels, fill_value=0)
    compatibility = {
        "samples": {"count": len(samples), "fingerprint": sample_fingerprint, "fingerprint_columns": fingerprint_columns},
        "posterior": {"artifact": "cache/alc_selector_posteriors.npy", "sample_fingerprint": sample_fingerprint, "shape": list(posterior.shape), "dtype": str(posterior.dtype), "sum_atol": float(alc_config["posterior_sum_atol"])},
        "candidates": {"artifact": "alc_label_candidates.csv", "sample_fingerprint": sample_fingerprint, "schema": candidate_columns, "only_prediction_divergences": True, "order": ["alc_phi descending", "sample_id ascending"]},
    }
    if CHECKPOINT_PATH.is_file():
        compatibility["model"] = {"artifact": "model/alc_vanilla_resnet50.pt", "architecture": alc_config["architecture"], "classes": coin_labels, "image_size": image_size, "training": dict(training_config), "load_validated": bool(model_validated_this_run)}
    write_manifest(
        STAGE, "06-active-label-cleaning.ipynb", inputs={"annotations_normalized": ANNOTATIONS_PATH},
        parameters={"seed": seed, "architecture": alc_config["architecture"], "image_size": image_size, "crop_padding_fraction": float(alc_config["crop_padding_fraction"]), "training": dict(training_config)},
        artifacts=artifacts,
        summary={"samples": len(samples), "candidates": len(candidates), "candidate_rate": len(candidates) / len(samples), "candidate_distribution_by_original_class": {str(k): int(v) for k, v in distribution.items()}, "confusion_matrix_original_by_predicted": {str(row): {str(column): int(confusion.loc[row, column]) for column in coin_labels} for row in coin_labels}, "mean_phi_candidates": float(candidates["alc_phi"].mean()), "maximum_phi": float(candidates["alc_phi"].max())},
        compatibility=compatibility, repo_root=ROOT,
    )

model_validated_this_run = False
if candidates_are_valid():
    candidates = pd.read_csv(CANDIDATES_PATH, dtype={"original_label": str, "predicted_label": str})
    if not HISTORY_PATH.is_file():
        valid_checkpoint, checkpoint = checkpoint_is_valid()
        if valid_checkpoint and checkpoint.get("history"):
            pd.DataFrame(checkpoint["history"]).to_csv(HISTORY_PATH, index=False)
    pipeline_state = "valid candidates loaded; all dependent computation skipped"
else:
    posterior = None
    if POSTERIOR_PATH.is_file():
        try:
            posterior = np.load(POSTERIOR_PATH, allow_pickle=False)
        except (OSError, ValueError):
            posterior = None
    if posterior is not None and posterior_is_valid(posterior):
        model_validated_this_run = model_attestation_is_current()
        pipeline_state = "valid posterior loaded; model work skipped"
    else:
        valid_checkpoint, checkpoint = checkpoint_is_valid()
        if valid_checkpoint:
            model_validated_this_run = True
            if not HISTORY_PATH.is_file() and checkpoint.get("history"):
                pd.DataFrame(checkpoint["history"]).to_csv(HISTORY_PATH, index=False)
            model = build_selector(); model.load_state_dict(checkpoint["model_state_dict"]); model.to(device)
            pipeline_state = "valid checkpoint loaded; inference executed"
        else:
            model, checkpoint = train_selector()
            model_validated_this_run = True
            pipeline_state = "selector trained; inference executed"
        posterior = infer(model)
    candidates, predicted_label = build_candidates(posterior)
    candidates.to_csv(CANDIDATES_PATH, index=False)
    save_manifest(posterior, candidates, predicted_label)

print(pipeline_state)
print("Review candidates:", len(candidates))
display(candidates.head(30))

In [ ]:
def show_candidates(n=12, full_image=False, columns=4):
    prepare_fast_image_access()
    chosen = candidates.head(n)
    rows = math.ceil(len(chosen) / columns)
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis in axes:
        axis.axis("off")
    for axis, (_, candidate) in zip(axes, chosen.iterrows()):
        row = samples.iloc[int(candidate["sample_id"])]
        with Image.open(ACTIVE_IMAGES_DIR / row["relative_path"]).convert("RGB") as image:
            shown = image.copy() if full_image else crop_bbox(image, row, float(alc_config["crop_padding_fraction"]))
        axis.imshow(shown)
        if full_image:
            axis.add_patch(Rectangle((row["bbox_xmin"], row["bbox_ymin"]), row["bbox_xmax"] - row["bbox_xmin"], row["bbox_ymax"] - row["bbox_ymin"], fill=False, linewidth=2))
        axis.set_title(f"rank={candidate['priority_rank']} | original={candidate['original_label']} | predicted={candidate['predicted_label']}\nconfidence={candidate['predicted_confidence']:.3f} | Phi={candidate['alc_phi']:.3f}")
    plt.tight_layout()
    plt.show()

# Visualization is optional and is the only reason to access images when candidates are already valid.
# show_candidates(n=12)
# show_candidates(n=8, full_image=True)

## Inspect an ALC Candidate

This section is self-contained: it can be run without executing the previous cells in this notebook.

Use the zero-based row index from `alc_label_candidates.csv` to display the model crop and the complete source image with its bounding box side by side. The function also prints the complete CSV row.


In [ ]:
# Standalone setup for candidate inspection
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    inspection_base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    inspection_base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

INSPECTION_ROOT = inspection_base_folder.resolve()
if str(INSPECTION_ROOT) not in sys.path:
    sys.path.insert(0, str(INSPECTION_ROOT))

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from PIL import Image

from curation.common import crop_bbox, load_config, prepare_dataset, stage_output_dir

inspection_config = load_config(INSPECTION_ROOT)
inspection_dataset = prepare_dataset(INSPECTION_ROOT)
inspection_images_dir = inspection_dataset["images_dir"]
inspection_candidates_path = (
    stage_output_dir("06-active-label-cleaning", INSPECTION_ROOT, create=False)
    / "alc_label_candidates.csv"
)


def show_label_candidate(csv_index):
    """Display an ALC crop and its source image at a CSV row.

    Args:
        csv_index: Zero-based row index in alc_label_candidates.csv.
    """
    rows = pd.read_csv(
        inspection_candidates_path,
        dtype={"original_label": str, "predicted_label": str},
    )
    if not isinstance(csv_index, (int, np.integer)):
        raise TypeError("csv_index must be an integer.")
    if csv_index < 0 or csv_index >= len(rows):
        raise IndexError(f"csv_index must be between 0 and {len(rows) - 1}.")

    candidate = rows.iloc[int(csv_index)]
    image_path = inspection_images_dir / candidate["relative_path"]

    with Image.open(image_path).convert("RGB") as image:
        full_image = image.copy()
        crop = crop_bbox(
            image,
            candidate,
            float(inspection_config["alc"]["crop_padding_fraction"]),
        )

    figure, axes = plt.subplots(1, 2, figsize=(13, 6))
    axes[0].imshow(crop)
    axes[0].set_title("ResNet-50 input crop")
    axes[0].axis("off")

    axes[1].imshow(full_image)
    axes[1].add_patch(
        Rectangle(
            (candidate["bbox_xmin"], candidate["bbox_ymin"]),
            candidate["bbox_xmax"] - candidate["bbox_xmin"],
            candidate["bbox_ymax"] - candidate["bbox_ymin"],
            fill=False,
            linewidth=2,
            edgecolor="red",
        )
    )
    axes[1].set_title("Complete image and annotation bounding box")
    axes[1].axis("off")

    figure.suptitle(
        f"CSV row {csv_index} | original={candidate['original_label']} | "
        f"predicted={candidate['predicted_label']} | Phi={candidate['alc_phi']:.3f}"
    )
    plt.tight_layout()
    plt.show()

    print(candidate.to_string())
    return candidate

# Example:
# show_label_candidate(0)

In [ ]:
show_label_candidate(0)

In [ ]:
show_label_candidate(2)